# 05 — Predictive Model (one section, not the centerpiece)

This notebook exists to check which factors carry predictive signal, using the competition's
own labels (`train_v2.csv`) — not to chase leaderboard AUC. Nearly every public repo on this
dataset stacks classifiers and reports AUC as the whole point; here it's the last section, and
its job is to tie back to the levers examined in notebook 04, not to stand alone.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, classification_report, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path.cwd().parent))
from src import data_load, plotting

plotting.set_style()

## Build the feature table
One row per `msno` in `train_v2.csv`, using each user's most recent transaction as of the labeling snapshot plus `members_v3` demographics. Only fields present in the three approved files are used — no `user_logs_v2.csv`.

In [ ]:
transactions = data_load.load_transactions()
members = data_load.load_members()
train = data_load.load_train()

latest_txn = (
    transactions.sort_values("transaction_date")
    .groupby("msno", observed=True)
    .tail(1)
    .set_index("msno")
)

features = (
    train.set_index("msno")
    .join(latest_txn, how="inner")
    .join(
        members.set_index("msno")[["city", "bd", "gender", "registered_via", "registration_init_time"]],
        how="left",
    )
    .reset_index()
)

with np.errstate(divide="ignore", invalid="ignore"):
    features["discount_depth"] = np.where(
        features["plan_list_price"] > 0,
        (features["plan_list_price"] - features["actual_amount_paid"]) / features["plan_list_price"],
        np.nan,
    )
features["age_valid"] = features["bd"].between(10, 90)
features["bd_clean"] = features["bd"].where(features["age_valid"])
features["tenure_days"] = (features["transaction_date"] - features["registration_init_time"]).dt.days
features["gender_known"] = features["gender"].notna().astype(int)

print(f"{len(features):,} labeled users with a matched transaction (of {len(train):,} in train_v2.csv)")

## Class balance and base rate — report this before any model metric

In [ ]:
base_rate = features["is_churn"].mean()
print(features["is_churn"].value_counts())
print(f"\nBase (churn) rate: {base_rate:.4f}  ->  a model that always predicts 'no churn' scores {1 - base_rate:.4f} accuracy.")
print("Accuracy is not reported anywhere below for exactly this reason — on data this imbalanced")
print("it's dominated by the majority-class guess and tells a business stakeholder nothing.")

## Train / test split and features

In [ ]:
feature_cols = [
    "payment_method_id", "payment_plan_days", "plan_list_price", "actual_amount_paid",
    "discount_depth", "is_auto_renew", "is_cancel", "city", "bd_clean", "gender_known",
    "registered_via", "tenure_days",
]
model_df = features[feature_cols + ["is_churn"]].copy()
model_df["discount_depth"] = model_df["discount_depth"].fillna(model_df["discount_depth"].median())
model_df["bd_clean"] = model_df["bd_clean"].fillna(model_df["bd_clean"].median())
model_df["tenure_days"] = model_df["tenure_days"].fillna(model_df["tenure_days"].median())

X = model_df[feature_cols]
y = model_df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f"train: {len(X_train):,} ({y_train.mean():.4f} churn rate)  |  test: {len(X_test):,} ({y_test.mean():.4f} churn rate)")

## Logistic regression baseline

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(max_iter=1000, class_weight="balanced")
logit.fit(X_train_scaled, y_train)
logit_scores = logit.predict_proba(X_test_scaled)[:, 1]

print(f"Logistic regression AUC: {roc_auc_score(y_test, logit_scores):.4f}")

## Random forest
Tree ensemble to check for non-linear / interaction signal beyond the linear baseline; also lets us use `shap.TreeExplainer` directly for feature attributions.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=50,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print(f"Random forest AUC: {roc_auc_score(y_test, rf_scores):.4f}")
print(f"Random forest average precision (PR-AUC): {average_precision_score(y_test, rf_scores):.4f}")
print(f"(compare against the base rate {y_test.mean():.4f} as the random-classifier floor for PR-AUC)")

## Precision / recall at a chosen threshold
No bare accuracy anywhere in this notebook — on data this imbalanced it's dominated by the majority class. Precision/recall at a threshold chosen for a stated business purpose is what's actionable.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, rf_scores)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, color=plotting.CATEGORICAL["blue"], linewidth=2)
ax.axhline(y_test.mean(), color=plotting.INK_MUTED, linestyle="--", linewidth=1, label="base rate")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall curve — random forest", loc="left", pad=12)
ax.legend()
fig.tight_layout()
fig.savefig("../figures/05_precision_recall.png", bbox_inches="tight")

In [ ]:
TARGET_FLAG_RATE = 0.10  # business constraint: retention outreach capacity
threshold = np.quantile(rf_scores, 1 - TARGET_FLAG_RATE)
predicted_positive = rf_scores >= threshold

p_at_k = precision_score(y_test, predicted_positive)
r_at_k = recall_score(y_test, predicted_positive)
print(f"Flagging the top {TARGET_FLAG_RATE:.0%} highest-risk users (score >= {threshold:.3f}):")
print(f"  precision = {p_at_k:.3f}  (of flagged users, this share actually churn)")
print(f"  recall    = {r_at_k:.3f}  (of all churners, this share get flagged)")
print()
print(classification_report(y_test, predicted_positive, target_names=["retained", "churned"]))

## SHAP feature attribution

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test, check_additivity=False)

if isinstance(shap_values, list):
    shap_for_churn = shap_values[1]
elif np.ndim(shap_values) == 3:
    shap_for_churn = shap_values[:, :, 1]
else:
    shap_for_churn = shap_values

shap.summary_plot(shap_for_churn, X_test, show=False)
plt.gcf().set_size_inches(8, 6)
plt.tight_layout()
plt.savefig("../figures/05_shap_summary.png", bbox_inches="tight")
plt.show()

## Tie back to section 04

Compare the top SHAP features to the levers examined in notebook 04
(`is_auto_renew`, `payment_plan_days`, `discount_depth`, payment method). Where they agree,
that's converging evidence — the descriptive association and the predictive signal point the
same direction. Where the model surfaces something notebook 04 didn't examine directly (e.g.
`tenure_days`, `city`), that's a lead for further descriptive analysis, not a new causal claim.
This is a predictive model, not a causal one — every selection-effect caveat from notebooks
03-04 still applies to how these features should be interpreted.

## Section summary

Report, in this order: class balance / base rate, logistic-regression AUC vs. random-forest
AUC, precision/recall at the chosen operating threshold, and the top SHAP drivers with their
link back to notebook 04's levers. This is deliberately the last analytical notebook, not the
first — the business framing from notebooks 02-04 should drive how these results get used, not
the other way around.